# Unveiling Bias: recovered analysis

Individual UCIL20421 coursework, University of Manchester (2024), by Selin Deniz İşken.

This clean copy preserves the recovered eight-profession calculations. Original submitted figures in `../assets/` include a later 14-profession version whose source was not recovered. The two versions must not be treated as fully interchangeable.

Install `../requirements.txt` in a separate environment first. Place `glove.6B.100d.txt` in `../data/` and launch Jupyter from the package root. No data downloads or package installations run inside this notebook.

The function named `weat_score` computes an unstandardized difference of mean cosine associations, not a formal WEAT effect size or significance test. Its historical name and chart labels are retained; see the README for interpretation. All saved outputs and execution counts have been cleared. No replacement figure assets were generated during portfolio preparation.

In [ ]:
# Recovered notebook originally included Kaggle environment boilerplate.
# Credit: https://github.com/Kaggle/docker-python
import numpy as np
import pandas as pd


In [ ]:
from pathlib import Path

# Accept either the repository root or notebooks/ as the working directory.
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
glove_path = project_root / 'data' / 'glove.6B.100d.txt'
if not glove_path.is_file():
    raise FileNotFoundError('Download and extract glove.6B.100d.txt into data/; see README.md.')


In [ ]:
import gensim
from gensim.models import KeyedVectors

# Load the 100-dimensional GloVe vectors
glove_file = str(glove_path)
model = KeyedVectors.load_word2vec_format(glove_file, no_header=True)


In [ ]:
# Target words
male_terms = ['he', 'him', 'his', 'man', 'male', 'boy', 'brother', 'father', 'son']
female_terms = ['she', 'her', 'hers', 'woman', 'female', 'girl', 'sister', 'mother', 'daughter']

# Attribute words (e.g., professions)
professions = ['doctor', 'nurse', 'engineer', 'teacher', 'scientist', 'receptionist', 'manager', 'clerk']


In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def get_similarity_matrix(target_words, attribute_words, model):
    similarity_matrix = np.zeros((len(target_words), len(attribute_words)))
    for i, target in enumerate(target_words):
        for j, attribute in enumerate(attribute_words):
            if target in model and attribute in model:
                similarity_matrix[i, j] = cosine_similarity(model[target], model[attribute])
            else:
                similarity_matrix[i, j] = np.nan  # Handle missing words
    return similarity_matrix


In [ ]:
target_words = male_terms + female_terms
similarity_matrix = get_similarity_matrix(target_words, professions, model)


In [ ]:
import pandas as pd

similarity_df = pd.DataFrame(similarity_matrix, index=target_words, columns=professions)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.heatmap(similarity_df, annot=True, cmap='coolwarm', center=0, cbar_kws={'label': 'Cosine Similarity'})
plt.title('Heatmap of Gender Associations in Word Embeddings')
plt.xlabel('Professions')
plt.ylabel('Gendered Terms')
plt.show()


In [ ]:
import numpy as np

def weat_score(target_X, target_Y, attribute_A, attribute_B, model):
    def mean_cosine_similarity(word_list1, word_list2):
        return np.mean([model.similarity(w1, w2) for w1 in word_list1 for w2 in word_list2])

    s_X = mean_cosine_similarity(target_X, attribute_A) - mean_cosine_similarity(target_X, attribute_B)
    s_Y = mean_cosine_similarity(target_Y, attribute_A) - mean_cosine_similarity(target_Y, attribute_B)
    return s_X - s_Y


## Historical zero-score example

The following early experiment passes the same list as attribute A and B. Its differences therefore cancel to zero. It is retained as recovered; the later distinct-set calculation below supplies the meaningful domain comparison.

In [ ]:
# Gendered terms
male_terms = ['he', 'him', 'his', 'man', 'male', 'boy', 'brother', 'father', 'son']
female_terms = ['she', 'her', 'hers', 'woman', 'female', 'girl', 'sister', 'mother', 'daughter']

# Example for professions
professions = ['doctor', 'nurse', 'engineer', 'teacher', 'scientist', 'receptionist', 'manager', 'clerk']
professions_score = weat_score(male_terms, female_terms, professions, professions, model)

personality_traits = ['assertive', 'caring', 'dominant', 'emotional', 'logical', 'nurturing', 'rational', 'sensitive']
traits_score = weat_score(male_terms, female_terms, personality_traits, personality_traits, model)

activities = ['cooking', 'driving', 'sewing', 'fishing', 'dancing', 'hunting', 'knitting', 'programming']
activities_score = weat_score(male_terms, female_terms, activities, activities, model)


In [ ]:
bias_scores = {
    'Professions': professions_score,
    'Personality Traits': traits_score,
    'Activities': activities_score
    # Add other domains as needed
}


In [ ]:
import pandas as pd

bias_df = pd.DataFrame(list(bias_scores.items()), columns=['Domain', 'Bias Score'])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.barplot(x='Bias Score', y='Domain', data=bias_df, palette='viridis')
plt.title('Average Bias Scores Across Different Domains')
plt.xlabel('WEAT Bias Score')
plt.ylabel('Domain')
plt.axvline(0, color='grey', linestyle='--')
plt.show()


## Distinct attribute sets: domain comparison

This later recovered cell uses separate A/B lists. Expected approximate results: professions 0.15055; personality traits 0.04577; activities 0.13903. These differ from the essay narrative (0.14, 0.08, 0.12). The score is unstandardized and conditional on the chosen word lists.

In [ ]:
# Import libraries
import gensim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import KeyedVectors

# Load the 100-dimensional GloVe vectors
glove_file = str(glove_path)
model = KeyedVectors.load_word2vec_format(glove_file, no_header=True)

# Define the WEAT score function
def weat_score(target_X, target_Y, attribute_A, attribute_B, model):
    def mean_cosine_similarity(word_list1, word_list2):
        similarities = [
            model.similarity(w1, w2)
            for w1 in word_list1
            for w2 in word_list2
            if w1 in model and w2 in model
        ]
        return np.mean(similarities) if similarities else 0

    s_X = mean_cosine_similarity(target_X, attribute_A) - mean_cosine_similarity(target_X, attribute_B)
    s_Y = mean_cosine_similarity(target_Y, attribute_A) - mean_cosine_similarity(target_Y, attribute_B)
    return s_X - s_Y

# Define target words (gendered terms)
male_terms = ['he', 'him', 'his', 'man', 'male', 'boy', 'brother', 'father', 'son']
female_terms = ['she', 'her', 'hers', 'woman', 'female', 'girl', 'sister', 'mother', 'daughter']

# Define domain-specific attribute words
professions_A = ['doctor', 'engineer', 'scientist', 'manager']
professions_B = ['nurse', 'teacher', 'receptionist', 'clerk']

traits_A = ['assertive', 'dominant', 'logical', 'rational']
traits_B = ['caring', 'emotional', 'nurturing', 'sensitive']

activities_A = ['hunting', 'fishing', 'driving', 'programming']
activities_B = ['cooking', 'dancing', 'knitting', 'sewing']

# Compute WEAT scores for each domain
professions_score = weat_score(male_terms, female_terms, professions_A, professions_B, model)
traits_score = weat_score(male_terms, female_terms, traits_A, traits_B, model)
activities_score = weat_score(male_terms, female_terms, activities_A, activities_B, model)

# Create a dictionary to store the results
bias_scores = {
    'Professions': professions_score,
    'Personality Traits': traits_score,
    'Activities': activities_score
}

# Convert to a Pandas DataFrame
bias_df = pd.DataFrame(list(bias_scores.items()), columns=['Domain', 'Bias Score'])

# Plot the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x='Bias Score', y='Domain', data=bias_df, palette='viridis')
plt.title('WEAT Bias Scores Across Different Domains')
plt.xlabel('WEAT Bias Score')
plt.ylabel('Domain')
plt.axvline(0, color='grey', linestyle='--')
plt.show()


In [ ]:
# Compute similarity matrix for traits
traits_matrix = get_similarity_matrix(male_terms + female_terms, traits_A + traits_B, model)

# Create DataFrame
traits_df = pd.DataFrame(traits_matrix, index=male_terms + female_terms, columns=traits_A + traits_B)

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(traits_df, annot=True, cmap='coolwarm', center=0, cbar_kws={'label': 'Cosine Similarity'})
plt.title('Heatmap of Gender Associations with Traits in Word Embeddings')
plt.xlabel('Traits')
plt.ylabel('Gendered Terms')
plt.show()


In [ ]:
# Combine all attribute sets
combined_attributes = professions_A + professions_B + traits_A + traits_B + activities_A + activities_B

# Compute similarity matrix
combined_matrix = get_similarity_matrix(male_terms + female_terms, combined_attributes, model)

# Create DataFrame
combined_df = pd.DataFrame(combined_matrix, index=male_terms + female_terms, columns=combined_attributes)

# Plot heatmap
plt.figure(figsize=(15, 10))
sns.heatmap(combined_df, annot=False, cmap='coolwarm', center=0, cbar_kws={'label': 'Cosine Similarity'})
plt.title('Combined Heatmap of Gender Associations Across Professions, Traits, and Activities')
plt.xlabel('Attributes')
plt.ylabel('Gendered Terms')
plt.show()


In [ ]:
# Flatten the similarity matrix for professions
similarity_values = similarity_matrix.flatten()

# Plot distribution
plt.figure(figsize=(10, 6))
sns.histplot(similarity_values, bins=20, kde=True, color='blue')
plt.title('Distribution of Cosine Similarities (Professions)')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Compute absolute values of similarity matrix
absolute_similarity_df = similarity_df.abs()

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(absolute_similarity_df, annot=True, cmap='Blues', cbar_kws={'label': 'Absolute Cosine Similarity'})
plt.title('Heatmap of Magnitudes of Gender Associations (Professions)')
plt.xlabel('Professions')
plt.ylabel('Gendered Terms')
plt.show()


In [ ]:
# Select a specific male and female term for comparison
selected_male_term = 'he'
selected_female_term = 'she'

# Extract cosine similarities for the selected terms
male_similarities = similarity_df.loc[selected_male_term]
female_similarities = similarity_df.loc[selected_female_term]

# Combine into a DataFrame for plotting
comparison_df = pd.DataFrame({
    'Attribute': professions,
    selected_male_term: male_similarities,
    selected_female_term: female_similarities
}).melt(id_vars='Attribute', var_name='Gendered Term', value_name='Cosine Similarity')

# Plot bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x='Cosine Similarity', y='Attribute', hue='Gendered Term', data=comparison_df, palette='muted')
plt.title(f'Cosine Similarities of "{selected_male_term}" vs. "{selected_female_term}" Across Professions')
plt.xlabel('Cosine Similarity')
plt.ylabel('Attribute')
plt.axvline(0, color='grey', linestyle='--')
plt.show()


In [ ]:
# Plot clustermap
sns.clustermap(similarity_df, cmap='coolwarm', center=0, figsize=(12, 10), cbar_kws={'label': 'Cosine Similarity'})
plt.title('Clustermap of Gender Associations with Professions')
plt.show()


## Radar comparison

This code covers eight professions. The final submitted radar asset covers fourteen. Radial tick labels are hidden in the historical code; use cosine values and the grouped comparison chart for precise interpretation.

In [ ]:
from math import pi

# Prepare data for radar chart
attributes = professions
radar_values = [similarity_df.loc['he'], similarity_df.loc['she']]

# Create radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
angles = [n / float(len(attributes)) * 2 * pi for n in range(len(attributes))]
angles += angles[:1]

for values, label in zip(radar_values, ['He', 'She']):
    values = values.tolist() + values.tolist()[:1]
    ax.plot(angles, values, label=label)
    ax.fill(angles, values, alpha=0.1)

ax.set_yticklabels([])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(attributes)
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
plt.title('Radar Chart of Gender Bias Across Professions')
plt.show()


## Exploratory PCA

A two-dimensional projection is a visual summary, not a significance test. All selected words were found during the recovery check. The inherited label construction assumes no missing words.

In [ ]:
from sklearn.decomposition import PCA

# Extract embeddings for selected words
words = male_terms + female_terms + professions
word_vectors = [model[word] for word in words if word in model]

# Apply PCA for dimensionality reduction
pca = PCA(n_components=2)
reduced_vectors = pca.fit_transform(word_vectors)

# Create a DataFrame for plotting
pca_df = pd.DataFrame(reduced_vectors, columns=['PC1', 'PC2'])
pca_df['Word'] = words[:len(reduced_vectors)]

# Plot the projections
plt.figure(figsize=(10, 8))
sns.scatterplot(x='PC1', y='PC2', data=pca_df, hue=pca_df['Word'].apply(lambda x: 'Male' if x in male_terms else 'Female' if x in female_terms else 'Profession'))
for i, word in enumerate(pca_df['Word']):
    plt.text(pca_df['PC1'][i], pca_df['PC2'][i], word, fontsize=9)
plt.title('PCA Projection of Gendered Terms and Professions')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()


## Exploratory network

The force-directed layout has no fixed seed, so positions can vary. All available gender–profession pairs are connected; node degree is largely determined by the two word-list sizes and is not evidence of bias strength. Edge-color normalization and labels are preserved from the recovered code; do not use this exploratory plot as a calibrated quantitative result.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Prepare the data for network visualization
edges = []
weights = []

# Use similarity matrix to create edges and weights
for i, target in enumerate(target_words):
    for j, attribute in enumerate(professions):
        if not np.isnan(similarity_matrix[i, j]):
            edges.append((target, attribute))
            weights.append(similarity_matrix[i, j])

# Normalize weights for better visualization
weights = np.array(weights)
weights_normalized = (weights - weights.min()) / (weights.max() - weights.min())

# Create a network graph
G = nx.Graph()

# Add edges and weights
for edge, weight in zip(edges, weights_normalized):
    G.add_edge(edge[0], edge[1], weight=weight)

# Create node colors based on node type
node_colors = []
for node in G.nodes():
    if node in male_terms:
        node_colors.append('blue')
    elif node in female_terms:
        node_colors.append('pink')
    else:
        node_colors.append('green')

# Set node sizes based on their degree (number of connections)
node_sizes = [1000 * G.degree(node) for node in G.nodes()]

# Draw the network graph
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5)  # Artistic, force-directed layout
edges = nx.draw_networkx_edges(
    G, pos, alpha=0.5, edge_color=weights_normalized, edge_cmap=plt.cm.coolwarm
)
nodes = nx.draw_networkx_nodes(
    G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.8
)
labels = nx.draw_networkx_labels(G, pos, font_size=10, font_color="black")

# Add color bar for weights
sm = plt.cm.ScalarMappable(cmap=plt.cm.coolwarm, norm=plt.Normalize(vmin=weights.min(), vmax=weights.max()))
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca())
cbar.set_label("Cosine Similarity", rotation=270, labelpad=20)

# Artistic styling
plt.title("The Web of Bias: Gender Associations in Word Embeddings", fontsize=16)
plt.axis('off')
plt.show()
